# import

In [1]:
import sys
import os, time, torch, tqdm, gc, numpy as np, pandas as pd

import torch.nn as nn
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch.amp import autocast, GradScaler         
from torch.nn.utils import clip_grad_norm_
import torch.nn.functional as F

from torch_geometric.utils import negative_sampling
from torch.optim.lr_scheduler import ReduceLROnPlateau

from neo4j.exceptions import ServiceUnavailable
from neomodel import config as neomodel_config
from neomodel import db

from app.config.local import NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD # src/app/config/local.py
NEO4J_USER     = NEO4J_USER
NEO4J_PWD      = DB_PASSWORD
NEO4J_HOST     = NEO4J_HOST
NEO4J_PORT     = NEO4J_PORT

neomodel_config.DATABASE_URL = f'bolt://{NEO4J_USER}:{NEO4J_PWD}@{NEO4J_HOST}:{NEO4J_PORT}'

try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공! → 응답값:", result[0][0])
except ServiceUnavailable as e:
    print("❌ Neo4j 연결 실패:", e)

✅ config_module: ['CARD_SERVER_ADDRESS', 'DB_HOST', 'DB_NAME', 'DB_PASSWORD', 'DB_PORT', 'DB_USER', 'DEBUG', 'GEMINI_API_KEY', 'GOOGLE_APPLICATION_CREDENTIALS', 'GOOGLE_PROJECT', 'NEO4J_HOST', 'NEO4J_PORT', 'NEO4J_USER', 'SWAGGER_SPECS_JSON_ROUTE', 'SWAGGER_SPECS_ROUTE', 'SWAGGER_STATIC_PATH', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']
✅ Neo4j 연결 성공! → 응답값: 1


In [2]:
# conver cypher to df(pandas)
def cypher_df(q: str, params: dict | None = None) -> pd.DataFrame:
    records, cols = db.cypher_query(q, params or {})
    df = pd.DataFrame(records, columns=cols)

    # 1) 어떤 컬럼을 변환할지 결정
    id_cols = [c for c in df.columns
               if c.endswith('_id') or c in {'mid', 'oid', 'pid'}]

    # 2) 캐스팅 (문자·공백·NULL → NaN → Int64 → int64)
    for col in id_cols:
        df[col] = (
            pd.to_numeric(df[col], errors="coerce")  # 문자열·공백 처리
              .astype("Int64")                      # Nullable 정수
              .astype("int64")                      # 확정 int64
        )
    return df

# enable safe mapping(drop rows when failure)
def safe_map(series: pd.Series, mapping: dict[int,int], col: str) -> np.ndarray:
    mapped = series.map(mapping)
    miss   = mapped.isna().sum()
    if miss:
        print(f"⚠️  {col}: {miss:,}개 매핑 실패 → 행 제거")
        mapped = mapped.dropna()
    return mapped.astype(np.int64).to_numpy()

def free_mem(*names):
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# data load

In [3]:
# HetSAGE 모델 정의
class HetSAGE(nn.Module):
    def __init__(self, hid: int = 64, out: int = 64):
        super().__init__()
        # ① 1-hop
        self.conv1 = HeteroConv({
            ('user',   'ordered',      'order'):   SAGEConv((-1, -1), hid),
            ('order',  'contains',     'product'): SAGEConv((-1, -1), hid),
            ('order',  'ordered_by',   'user'):    SAGEConv((-1, -1), hid),  # 역
            ('product','contained_in', 'order'):   SAGEConv((-1, -1), hid)   # 역
        }, aggr='mean')
        # ② 2-hop
        self.conv2 = HeteroConv({
            ('user',   'ordered',      'order'):   SAGEConv((hid, hid), out),
            ('order',  'contains',     'product'): SAGEConv((hid, hid), out),
            ('order',  'ordered_by',   'user'):    SAGEConv((hid, hid), out),
            ('product','contained_in', 'order'):   SAGEConv((hid, hid), out)
        }, aggr='mean')

    def forward(self, x_dict, edge_dict):
        h = {k: torch.relu(v) for k, v in self.conv1(x_dict, edge_dict).items()}
        return self.conv2(h, edge_dict)                 # h_dict ⟶ 최종 임베딩

In [4]:
# 1. 체크포인트 경로
ckpt_path = "checkpoints/HetSAGE_ep100_20250524_085234.pt"  # 실제 파일명 확인 (확장자 필수!)

# 3. 체크포인트 불러오기
print(f"📂 체크포인트 로드 중... ({ckpt_path})")
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only = False)
print(f"✔️  체크포인트 로드 완료: Epoch {ckpt.get('epoch', 'N/A')}, Precision(best): {ckpt.get('precision', 'N/A')}")

# 4. 모델 하이퍼파라미터 복원 및 모델 생성
hid_dim = ckpt.get('hid_dim', 64)
out_dim = ckpt.get('out_dim', 64)
num_layers = ckpt.get('num_layers', 2)
print(f"⚙️  HetSAGE 모델 복원 (hidden: {hid_dim}, out: {out_dim}, layers: {num_layers})")
model = HetSAGE(hid=hid_dim, out=out_dim)    # 구조가 다르면 에러!
model.load_state_dict(ckpt['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

# 5. 인덱스 매핑 복원
m2i = ckpt['m2i']
o2i = ckpt['o2i']
p2i = ckpt['p2i']
print(f"🔗 인덱스 매핑 복원 (user: {len(m2i)}, order: {len(o2i)}, product: {len(p2i)})")


📂 체크포인트 로드 중... (checkpoints/HetSAGE_ep100_20250524_085234.pt)
✔️  체크포인트 로드 완료: Epoch 100, Precision(best): N/A
⚙️  HetSAGE 모델 복원 (hidden: 64, out: 64, layers: 2)
🔗 인덱스 매핑 복원 (user: 206209, order: 3346083, product: 49685)


In [5]:
# 2-1. Order feature  ────────────────────────────
df_order_feat = cypher_df("""
MATCH (o:Order)
RETURN o.order_id AS oid,
       o.days_since_prior_order_norm AS dsp,
       o.order_dow_norm              AS dow,
       o.order_hour_of_day_norm      AS hr,
       o.order_number_norm           AS idx
""")

# ※ 매핑 실패 행 제거 (oid → o2i)
df_order_feat["oid_idx"] = safe_map(df_order_feat.oid, o2i, "oid")
order_feat = (
    df_order_feat
      .sort_values("oid_idx")[["dsp","dow","hr","idx"]]
      .to_numpy(np.float32)
)                              

# 2-2. Product feature  ──────────────────────────
df_prod_vec = cypher_df("""
MATCH (p:Product)
RETURN p.product_id AS pid,
       p.category_vector + p.name_vector AS vec
""")

df_prod_vec["pid_idx"] = safe_map(df_prod_vec.pid, p2i, "pid")
prod_feat = torch.tensor(
    np.stack(df_prod_vec.sort_values("pid_idx").vec.values),
    dtype=torch.float32
)                                             # shape = [N_prod, D]
prod_feat_dim = prod_feat.size(1)

print(f"Order feature  : {order_feat.shape}")
print(f"Product vector : {prod_feat.shape}")


Order feature  : (3346083, 4)
Product vector : torch.Size([49685, 2048])


In [6]:
# ────────────────────────────────────────────────────────────────
# 0. 테스트용 prior/label 간선 및 gt_dict 생성 (negative X)
# ────────────────────────────────────────────────────────────────

# 1) prior 간선 (eval_set='test', role_test='prior')
df_edge_te = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid"))
), 0).to(torch.long)
op_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge_te.pid, p2i, "pid"))
), 0).to(torch.long)

# 3) 테스트용 HeteroData
test_data = HeteroData()
test_data['user'].num_nodes    = len(m2i)
test_data['order'].num_nodes   = len(o2i)
test_data['product'].num_nodes = len(p2i)
test_data['user'].x    = nn.Embedding(len(m2i), 32).weight
test_data['order'].x   = torch.tensor(order_feat, dtype=torch.float32)
test_data['product'].x = prod_feat
test_data['user','ordered','order'].edge_index         = uo_te
test_data['order','ordered_by','user'].edge_index      = uo_te.flip(0)
test_data['order','contains','product'].edge_index     = op_te
test_data['product','contained_in','order'].edge_index = op_te.flip(0)

In [7]:
# 1. ground-truth: 각 test user의 정답 product 인덱스 리스트
df_gt = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='train'
RETURN m.member_id AS mid, collect(p.product_id) AS gt_list
""")

mid_arr = safe_map(df_gt.mid, m2i, "mid")
# 각 gt_list가 파이썬 리스트(list) 형태 → 각 리스트별로 개별적으로 매핑
gt_list_arr = [
    [p2i[pid] for pid in pid_list if pid in p2i]
    for pid_list in df_gt.gt_list
]
gt_dict = {
    int(mid): [int(pid) for pid in pid_list]
    for mid, pid_list in zip(mid_arr, gt_list_arr) if mid != -1
}
print(f"GT users (index): {len(gt_dict):,}")


GT users (index): 75,000


In [8]:
free_mem('df_order_feat', 'df_prod_vec', 'df_edge_te', 'df_gt')

In [9]:
test_data = test_data.to('cpu')

# Test

In [17]:
def build_subgraph_for_users(user_ids, uo_te, op_te, order_feat, prod_feat, m2i, o2i, p2i, gt_dict):
    import torch
    from torch_geometric.data import HeteroData

    user_idx = np.array([m2i[u] for u in user_ids])
    mask_user = np.isin(uo_te[0].cpu().numpy(), user_idx)
    uo_sub = uo_te[:, mask_user]
    order_idx = np.unique(uo_sub[1].cpu().numpy())
    mask_order = np.isin(op_te[0].cpu().numpy(), order_idx)
    op_sub = op_te[:, mask_order]
    product_idx = np.unique(op_sub[1].cpu().numpy())

    old2new_user = {old: new for new, old in enumerate(user_idx)}
    old2new_order = {old: new for new, old in enumerate(order_idx)}
    old2new_prod = {old: new for new, old in enumerate(product_idx)}

    uo_sub_remap = torch.stack([
        torch.tensor([old2new_user[u.item()] for u in uo_sub[0]]),
        torch.tensor([old2new_order[o.item()] for o in uo_sub[1]])
    ], 0)
    op_sub_remap = torch.stack([
        torch.tensor([old2new_order[o.item()] for o in op_sub[0]]),
        torch.tensor([old2new_prod[p.item()] for p in op_sub[1]])
    ], 0)

    order_x = torch.tensor(np.stack([order_feat[o] for o in order_idx]), dtype=torch.float32)
    product_x = torch.tensor(np.stack([prod_feat[p] for p in product_idx]), dtype=torch.float32)

    gt_dict_remap = {
        old2new_user[m2i[u]]: [old2new_prod[p] for p in gt_dict[u] if p in old2new_prod]
        for u in user_ids if m2i[u] in old2new_user
    }

    data = HeteroData()
    data['user'].num_nodes = len(user_idx)
    data['order'].num_nodes = len(order_idx)
    data['product'].num_nodes = len(product_idx)
    data['user'].x = torch.nn.Embedding(len(user_idx), 32).weight
    data['order'].x = order_x
    data['product'].x = product_x
    data['user','ordered','order'].edge_index = uo_sub_remap
    data['order','ordered_by','user'].edge_index = uo_sub_remap.flip(0)
    data['order','contains','product'].edge_index = op_sub_remap
    data['product','contained_in','order'].edge_index = op_sub_remap.flip(0)

    return data, gt_dict_remap


### Memory Check

In [10]:
if torch.cuda.is_available():
    print(f"GPU Total Mem: {torch.cuda.get_device_properties(0).total_memory/1024/1024:.2f} MB")
    print(f"GPU Allocated: {torch.cuda.memory_allocated()/1024/1024:.2f} MB")
    print(f"GPU Cached   : {torch.cuda.memory_reserved()/1024/1024:.2f} MB")

GPU Total Mem: 7940.31 MB
GPU Allocated: 1.15 MB
GPU Cached   : 2.00 MB


### 1

In [ ]:
### checkpoints/HetSAGE_ep100_20250524_085234.pt" : 드롭아웃 없이 학습한 첫 번째 모델 

In [18]:
@torch.no_grad()
def validate_minibatch(model, user_ids, uo_te, op_te, order_feat, prod_feat, m2i, o2i, p2i, gt_dict, TOP_K=5, BATCH_SIZE=1000):
    model.eval()
    precs = []
    CHUNK = 200_000
    
    total_batches = int(np.ceil(len(user_ids) / BATCH_SIZE))
    
    for batch_num in tqdm.tqdm(range(total_batches), desc="🔎 미니배치 평가"):
        batch_user_ids = user_ids[batch_num * BATCH_SIZE : (batch_num + 1) * BATCH_SIZE]

        batch_data, gt_dict_remap = build_subgraph_for_users(
            batch_user_ids, uo_te, op_te, order_feat, prod_feat, m2i, o2i, p2i, gt_dict
        )

        batch_data = batch_data.to(device)
        z = model(batch_data.x_dict, batch_data.edge_index_dict)
        uemb = z['user'].cpu()
        pemb = z['product'].cpu()

        # GPU 메모리 즉시 정리
        del batch_data, z
        torch.cuda.empty_cache()

        for uid in range(len(batch_user_ids)):
            if uid not in gt_dict_remap:
                continue
            gt = gt_dict_remap[uid]
            u = uemb[uid]

            scores_all, idx_all = [], []
            for s in range(0, len(pemb), CHUNK):
                sc, ix = (pemb[s:s+CHUNK] @ u).topk(TOP_K)
                scores_all.append(sc)
                idx_all.append(ix + s)

            scores_all = torch.cat(scores_all)
            idx_all = torch.cat(idx_all)
            topk_idx = scores_all.topk(TOP_K).indices
            top_pids = idx_all[topk_idx].numpy()

            hit = len(set(top_pids) & set(gt))
            precs.append(hit / TOP_K)

        # 각 배치 완료 후 메모리 정리
        del uemb, pemb, scores_all, idx_all, topk_idx, top_pids
        gc.collect()
        torch.cuda.empty_cache()

    return float(np.mean(precs))

# ────────────────────────────────────────────
# 🚩 평가를 위한 user id 준비 (gt_dict 이미 존재한다고 가정)
# ────────────────────────────────────────────
test_user_ids = list(gt_dict.keys())  # 전체 75,000명 유저 리스트
print(f"🧑 전체 테스트 유저 수: {len(test_user_ids):,}")

# 평가 시작
t0 = time.perf_counter()
precision = validate_minibatch(
    model=model,
    user_ids=test_user_ids,
    uo_te=uo_te,
    op_te=op_te,
    order_feat=order_feat,
    prod_feat=prod_feat,
    m2i=m2i,
    o2i=o2i,
    p2i=p2i,
    gt_dict=gt_dict,
    TOP_K=5,
    BATCH_SIZE=1000
)

print(f"\n🎯 최종 Precision@5 (75000명 테스트): {precision:.4f}")
print(f"⏱️ 총 평가 시간: {time.perf_counter()-t0:.2f}초")


🧑 전체 테스트 유저 수: 75,000


🔎 미니배치 평가: 100%|█████████████████████████| 75/75 [00:55<00:00,  1.36it/s]


🎯 최종 Precision@5 (75000명 테스트): 0.0000
⏱️ 총 평가 시간: 55.07초


In [ ]:
TOP_K = 5 # evaluation count
@torch.no_grad()
def evaluate(model, test_data, gt_dict, m2i, p2i, k=5):
    model.eval()
    print("🚦 평가 시작: 모델 임베딩 추출 중...")
    test_data = test_data.to(device)
    z = model(test_data.x_dict, test_data.edge_index_dict)
    uemb = z['user'].cpu()
    pemb = z['product'].cpu()
    print("✅ 임베딩 추출 완료")

    CHUNK = 200_000
    precs = []
    total = len(gt_dict)
    for step, (mid, gt) in enumerate(gt_dict.items()):
        if mid not in m2i:
            print(f"⚠️  유저 {mid} m2i 매핑 누락, 건너뜀")
            continue
        u = uemb[m2i[mid]]
        # TopK 추천
        scores_all, idx_all = [], []
        for s in range(0, len(pemb), CHUNK):
            sc, ix = (pemb[s:s+CHUNK] @ u).topk(k)
            scores_all.append(sc)
            idx_all.append(ix + s)
        scores_all = torch.cat(scores_all)
        idx_all    = torch.cat(idx_all)
        topk_idx   = scores_all.topk(k).indices
        top_pred_pidx = idx_all[topk_idx].numpy()
        # 추천 결과를 실제 product_id로 매핑
        pred_products = [list(p2i.keys())[list(p2i.values()).index(p)] for p in top_pred_pidx]
        true_products = [list(p2i.keys())[p] if isinstance(p, int) else p for p in gt]
        hit = len(set(pred_products) & set(true_products))
        precs.append(hit / k)
        if (step + 1) % 1000 == 0 or (step + 1) == total:
            print(f"  [{step+1}/{total}] 유저 평가 중 (현재 평균 P@{k}: {np.mean(precs):.4f})")
    return float(np.mean(precs))


    
    

# 7. 평가 실행
t0 = time.perf_counter()
P_at_K = evaluate(model, test_data, gt_dict, m2i, p2i, k=TOP_K)
print(f"🏁 최종 Precision@{TOP_K}: {P_at_K:.4f} ({time.perf_counter()-t0:.1f}s 경과)")


### 2

In [ ]:
# HetSAGE 모델 (dropout 적용)
class HetSAGE(nn.Module):
    def __init__(self, hid=128, out=128, p_drop=0.2, aggr='sum'):
        super().__init__()
        self.conv1 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
        }, aggr=aggr)
        self.conv2 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((hid, hid), out, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
        }, aggr=aggr)
        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(out)
        self.p_drop = p_drop

    def _drop(self, h: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        return {k: F.dropout(v, p=self.p_drop, training=self.training) for k, v in h.items()}

    def forward(self, x_dict, edge_index_dict):
        h = self.conv1(x_dict, edge_index_dict)
        h = {k: F.relu(self.norm1(v)) for k, v in h.items()}
        h = self._drop(h)
        h = self.conv2(h, edge_index_dict)
        h = {k: F.relu(self.norm2(v)) for k, v in h.items()}
        h = self._drop(h)
        return h

In [ ]:
# 1. 체크포인트 경로
ckpt_path = "checkpoints/HetSAGE_ep100_20250524_085234.pt"  # 실제 파일명 확인 (확장자 필수!)

# 3. 체크포인트 불러오기
print(f"📂 체크포인트 로드 중... ({ckpt_path})")
ckpt = torch.load(ckpt_path, map_location='cpu')
print(f"✔️  체크포인트 로드 완료: Epoch {ckpt.get('epoch', 'N/A')}, Precision(best): {ckpt.get('precision', 'N/A')}")

# 4. 모델 하이퍼파라미터 복원 및 모델 생성
hid_dim = ckpt.get('hid_dim', 64)
out_dim = ckpt.get('out_dim', 64)
num_layers = ckpt.get('num_layers', 2)
print(f"⚙️  HetSAGE 모델 복원 (hidden: {hid_dim}, out: {out_dim}, layers: {num_layers})")
model = HetSAGE(hid=hid_dim, out=out_dim)    # 구조가 다르면 에러!
model.load_state_dict(ckpt['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

# 5. 인덱스 매핑 복원
m2i = ckpt['m2i']
o2i = ckpt['o2i']
p2i = ckpt['p2i']
print(f"🔗 인덱스 매핑 복원 (user: {len(m2i)}, order: {len(o2i)}, product: {len(p2i)})")


@torch.no_grad()
def evaluate(model, test_data, gt_dict, m2i, p2i, k=5):
    model.eval()
    print("🚦 평가 시작: 모델 임베딩 추출 중...")
    test_data = test_data.to(device)
    z = model(test_data.x_dict, test_data.edge_index_dict)
    uemb = z['user'].cpu()
    pemb = z['product'].cpu()
    print("✅ 임베딩 추출 완료")

    CHUNK = 200_000
    precs = []
    total = len(gt_dict)
    for step, (mid, gt) in enumerate(gt_dict.items()):
        if mid not in m2i:
            print(f"⚠️  유저 {mid} m2i 매핑 누락, 건너뜀")
            continue
        u = uemb[m2i[mid]]
        # TopK 추천
        scores_all, idx_all = [], []
        for s in range(0, len(pemb), CHUNK):
            sc, ix = (pemb[s:s+CHUNK] @ u).topk(k)
            scores_all.append(sc)
            idx_all.append(ix + s)
        scores_all = torch.cat(scores_all)
        idx_all    = torch.cat(idx_all)
        topk_idx   = scores_all.topk(k).indices
        top_pred_pidx = idx_all[topk_idx].numpy()
        # 추천 결과를 실제 product_id로 매핑
        pred_products = [list(p2i.keys())[list(p2i.values()).index(p)] for p in top_pred_pidx]
        true_products = [list(p2i.keys())[p] if isinstance(p, int) else p for p in gt]
        hit = len(set(pred_products) & set(true_products))
        precs.append(hit / k)
        if (step + 1) % 1000 == 0 or (step + 1) == total:
            print(f"  [{step+1}/{total}] 유저 평가 중 (현재 평균 P@{k}: {np.mean(precs):.4f})")
    return float(np.mean(precs))

# 7. 평가 실행
t0 = time.perf_counter()
P_at_K = evaluate(model, test_data, gt_dict, m2i, p2i, k=TOP_K)
print(f"🏁 최종 Precision@{TOP_K}: {P_at_K:.4f} ({time.perf_counter()-t0:.1f}s 경과)")


### 3

In [ ]:
# HetSAGE 모델 (dropout 적용)
class HetSAGE(nn.Module):
    def __init__(self, hid=128, out=128, p_drop=0.2, aggr='sum'):
        super().__init__()
        self.conv1 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
        }, aggr=aggr)
        self.conv2 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((hid, hid), out, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
        }, aggr=aggr)
        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(out)
        self.p_drop = p_drop

    def _drop(self, h: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        return {k: F.dropout(v, p=self.p_drop, training=self.training) for k, v in h.items()}

    def forward(self, x_dict, edge_index_dict):
        h = self.conv1(x_dict, edge_index_dict)
        h = {k: F.relu(self.norm1(v)) for k, v in h.items()}
        h = self._drop(h)
        h = self.conv2(h, edge_index_dict)
        h = {k: F.relu(self.norm2(v)) for k, v in h.items()}
        h = self._drop(h)
        return h

In [ ]:
import torch
import numpy as np
import time
import os

# 1. 체크포인트 경로
ckpt_path = "checkpoints/HetSAGE_ep100_20250524_085234.pt"  # 실제 파일명 확인 (확장자 필수!)

# 2. HetSAGE 클래스는 반드시 import/정의되어 있어야 함
# from model import HetSAGE
# class HetSAGE(nn.Module): ...

# 3. 체크포인트 불러오기
print(f"📂 체크포인트 로드 중... ({ckpt_path})")
ckpt = torch.load(ckpt_path, map_location='cpu')
print(f"✔️  체크포인트 로드 완료: Epoch {ckpt.get('epoch', 'N/A')}, Precision(best): {ckpt.get('precision', 'N/A')}")

# 4. 모델 하이퍼파라미터 복원 및 모델 생성
hid_dim = ckpt.get('hid_dim', 64)
out_dim = ckpt.get('out_dim', 64)
num_layers = ckpt.get('num_layers', 2)
print(f"⚙️  HetSAGE 모델 복원 (hidden: {hid_dim}, out: {out_dim}, layers: {num_layers})")
model = HetSAGE(hid=hid_dim, out=out_dim)    # 구조가 다르면 에러!
model.load_state_dict(ckpt['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

# 5. 인덱스 매핑 복원
m2i = ckpt['m2i']
o2i = ckpt['o2i']
p2i = ckpt['p2i']
print(f"🔗 인덱스 매핑 복원 (user: {len(m2i)}, order: {len(o2i)}, product: {len(p2i)})")

# 6. 테스트 서브그래프/정답 (test_data, gt_dict)은 기존 방식대로 생성돼 있어야 함
# test_data = ...  # HeteroData, 기존 방식
# gt_dict = ...    # {user_idx: [product_idx,...]}

@torch.no_grad()
def evaluate(model, test_data, gt_dict, m2i, p2i, k=5):
    model.eval()
    print("🚦 평가 시작: 모델 임베딩 추출 중...")
    test_data = test_data.to(device)
    z = model(test_data.x_dict, test_data.edge_index_dict)
    uemb = z['user'].cpu()
    pemb = z['product'].cpu()
    print("✅ 임베딩 추출 완료")

    CHUNK = 200_000
    precs = []
    total = len(gt_dict)
    for step, (mid, gt) in enumerate(gt_dict.items()):
        if mid not in m2i:
            print(f"⚠️  유저 {mid} m2i 매핑 누락, 건너뜀")
            continue
        u = uemb[m2i[mid]]
        # TopK 추천
        scores_all, idx_all = [], []
        for s in range(0, len(pemb), CHUNK):
            sc, ix = (pemb[s:s+CHUNK] @ u).topk(k)
            scores_all.append(sc)
            idx_all.append(ix + s)
        scores_all = torch.cat(scores_all)
        idx_all    = torch.cat(idx_all)
        topk_idx   = scores_all.topk(k).indices
        top_pred_pidx = idx_all[topk_idx].numpy()
        # 추천 결과를 실제 product_id로 매핑
        pred_products = [list(p2i.keys())[list(p2i.values()).index(p)] for p in top_pred_pidx]
        true_products = [list(p2i.keys())[p] if isinstance(p, int) else p for p in gt]
        hit = len(set(pred_products) & set(true_products))
        precs.append(hit / k)
        if (step + 1) % 1000 == 0 or (step + 1) == total:
            print(f"  [{step+1}/{total}] 유저 평가 중 (현재 평균 P@{k}: {np.mean(precs):.4f})")
    return float(np.mean(precs))

# 7. 평가 실행
t0 = time.perf_counter()
P_at_K = evaluate(model, test_data, gt_dict, m2i, p2i, k=TOP_K)
print(f"🏁 최종 Precision@{TOP_K}: {P_at_K:.4f} ({time.perf_counter()-t0:.1f}s 경과)")
